# Custom Neo4j & PostgreSQL Generators (Experimental)

Define ad-hoc graph structures and relational tables via JSON — no server code changes needed.

**Generator types available for property/column values:**

| Generator | Description | Key Options |
|---|---|---|
| `uuid` | UUID v4 | — |
| `sequence` | Sequential IDs | `prefix`, `width` |
| `choice` | Pick from list | `values`, `weights` |
| `range_int` | Random integer | `min`, `max` |
| `range_float` | Random decimal | `min`, `max`, `precision` |
| `bool` | Random boolean | `probability` |
| `date` | Random date | `start`, `end`, `format` |
| `timestamp` | Random timestamp | `start`, `end` |
| `name` | Realistic full name | — |
| `email` | Email address | `domain` |
| `phone` | US phone number | — |
| `address` | Street address | — |
| `constant` | Fixed value | `value` |
| `null_or` | Nullable wrapper | `null_percent`, `rule` |

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

In [ ]:
%run "./_config"

In [ ]:
import neo4j
import time

driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **kwargs):
    with driver.session() as session:
        result = session.run(query, **kwargs)
        return [record.data() for record in result]

print(run_cypher("RETURN 1 AS test"))

---
# Part 1: Custom Neo4j Graphs

---
## Example 1: Organization Chart

Employees, departments, offices, and reporting relationships.

In [ ]:
org_chart = requests.post(f"{API_URL}/data-sources/neo4j/custom/start", headers=HEADERS, json={
    "name": "org_chart",
    "clear_before": True,
    "nodes": [
        {"label": "Employee", "count": 500, "properties": [
            {"name": "emp_id", "generator_rule": {"generator": "sequence", "prefix": "EMP-", "width": 5}},
            {"name": "name", "generator_rule": {"generator": "name"}},
            {"name": "department", "generator_rule": {"generator": "choice",
                "values": ["Engineering", "Sales", "Marketing", "HR", "Finance", "Legal", "Operations", "Product"]}},
            {"name": "title", "generator_rule": {"generator": "choice",
                "values": ["Analyst", "Associate", "Manager", "Senior Manager", "Director", "VP", "Engineer", "Senior Engineer", "Lead"],
                "weights": [4, 4, 3, 2, 1, 1, 4, 3, 2]}},
            {"name": "salary", "generator_rule": {"generator": "range_int", "min": 45000, "max": 250000}},
            {"name": "hire_date", "generator_rule": {"generator": "date", "start": "2015-01-01", "end": "2025-06-01"}},
            {"name": "email", "generator_rule": {"generator": "email", "domain": "acme-corp.com"}},
            {"name": "remote", "generator_rule": {"generator": "bool", "probability": 0.35}},
        ]},
        {"label": "Office", "count": 8, "properties": [
            {"name": "office_id", "generator_rule": {"generator": "sequence", "prefix": "OFF-", "width": 3}},
            {"name": "city", "generator_rule": {"generator": "choice",
                "values": ["New York", "San Francisco", "Chicago", "Austin", "Denver", "Seattle", "Boston", "Atlanta"]}},
            {"name": "capacity", "generator_rule": {"generator": "range_int", "min": 50, "max": 500}},
        ]},
        {"label": "Project", "count": 30, "properties": [
            {"name": "project_id", "generator_rule": {"generator": "sequence", "prefix": "PRJ-", "width": 4}},
            {"name": "name", "generator_rule": {"generator": "choice",
                "values": ["Platform Rebuild", "Mobile App v3", "Data Pipeline", "ML Infra", "Security Audit",
                           "CRM Integration", "API Gateway", "Analytics Dashboard", "Cloud Migration", "Cost Optimization"]}},
            {"name": "status", "generator_rule": {"generator": "choice",
                "values": ["active", "completed", "planning", "on_hold"], "weights": [5, 2, 2, 1]}},
            {"name": "budget", "generator_rule": {"generator": "range_int", "min": 50000, "max": 5000000}},
        ]},
    ],
    "relationships": [
        {"type": "WORKS_IN", "from_label": "Employee", "to_label": "Office",
         "probability": 0.2, "max_per_source": 1},
        {"type": "REPORTS_TO", "from_label": "Employee", "to_label": "Employee",
         "probability": 0.004, "max_per_source": 1},
        {"type": "ASSIGNED_TO", "from_label": "Employee", "to_label": "Project",
         "probability": 0.02,
         "properties": [{"name": "role", "generator_rule": {"generator": "choice",
             "values": ["lead", "contributor", "reviewer"]}}]},
    ],
}).json()

org_job_id = org_chart["job_id"]
print(org_chart)

In [ ]:
# Wait and check status
time.sleep(10)
status = requests.get(f"{API_URL}/data-sources/neo4j/custom/{org_job_id}", headers=HEADERS).json()
print(status)

In [ ]:
# Query the org chart
results = run_cypher("""
    MATCH (e:Employee)-[:WORKS_IN]->(o:Office)
    RETURN o.city AS office, count(e) AS employees,
           collect(DISTINCT e.department) AS departments
    ORDER BY employees DESC
""")
display(spark.createDataFrame(results))

In [ ]:
# Reporting chains
results = run_cypher("""
    MATCH path = (e:Employee)-[:REPORTS_TO*1..4]->(mgr:Employee)
    RETURN e.name AS employee, e.title AS title,
           length(path) AS chain_depth,
           mgr.name AS reports_to, mgr.title AS manager_title
    ORDER BY chain_depth DESC
    LIMIT 20
""")
if results:
    display(spark.createDataFrame(results))
else:
    print("No reporting chains found")

In [ ]:
# Project staffing
results = run_cypher("""
    MATCH (e:Employee)-[a:ASSIGNED_TO]->(p:Project)
    RETURN p.name AS project, p.status AS status, p.budget AS budget,
           count(e) AS team_size,
           collect(DISTINCT a.role) AS roles
    ORDER BY team_size DESC
""")
if results:
    display(spark.createDataFrame(results))
else:
    print("No project assignments found")

---
## Example 2: Supply Chain Network

Suppliers, warehouses, products, and shipping routes.

In [ ]:
supply_chain = requests.post(f"{API_URL}/data-sources/neo4j/custom/start", headers=HEADERS, json={
    "name": "supply_chain",
    "clear_before": True,
    "nodes": [
        {"label": "Supplier", "count": 100, "properties": [
            {"name": "supplier_id", "generator_rule": {"generator": "sequence", "prefix": "SUP-", "width": 4}},
            {"name": "name", "generator_rule": {"generator": "name"}},
            {"name": "country", "generator_rule": {"generator": "choice",
                "values": ["US", "China", "Germany", "Japan", "Mexico", "Canada", "UK", "India", "South Korea", "Taiwan"]}},
            {"name": "reliability_score", "generator_rule": {"generator": "range_float", "min": 60.0, "max": 99.9, "precision": 1}},
            {"name": "lead_time_days", "generator_rule": {"generator": "range_int", "min": 1, "max": 90}},
        ]},
        {"label": "Warehouse", "count": 20, "properties": [
            {"name": "warehouse_id", "generator_rule": {"generator": "sequence", "prefix": "WH-", "width": 3}},
            {"name": "location", "generator_rule": {"generator": "choice",
                "values": ["Los Angeles", "Chicago", "Dallas", "Newark", "Memphis", "Louisville"]}},
            {"name": "capacity_sqft", "generator_rule": {"generator": "range_int", "min": 10000, "max": 500000}},
            {"name": "utilization_pct", "generator_rule": {"generator": "range_float", "min": 30.0, "max": 98.0}},
        ]},
        {"label": "Product", "count": 500, "properties": [
            {"name": "sku", "generator_rule": {"generator": "sequence", "prefix": "SKU-", "width": 6}},
            {"name": "category", "generator_rule": {"generator": "choice",
                "values": ["Electronics", "Apparel", "Food", "Automotive", "Pharma", "Industrial"]}},
            {"name": "unit_cost", "generator_rule": {"generator": "range_float", "min": 0.50, "max": 5000.0}},
            {"name": "weight_kg", "generator_rule": {"generator": "range_float", "min": 0.01, "max": 500.0}},
            {"name": "hazardous", "generator_rule": {"generator": "bool", "probability": 0.08}},
        ]},
    ],
    "relationships": [
        {"type": "SUPPLIES", "from_label": "Supplier", "to_label": "Product", "probability": 0.02,
         "properties": [{"name": "contract_price", "generator_rule": {"generator": "range_float", "min": 1.0, "max": 5000.0}}]},
        {"type": "STORED_IN", "from_label": "Product", "to_label": "Warehouse", "probability": 0.05,
         "properties": [{"name": "quantity", "generator_rule": {"generator": "range_int", "min": 10, "max": 10000}}]},
        {"type": "SHIPS_TO", "from_label": "Warehouse", "to_label": "Warehouse", "probability": 0.08,
         "properties": [{"name": "transit_days", "generator_rule": {"generator": "range_int", "min": 1, "max": 7}}]},
    ],
}).json()

supply_job_id = supply_chain["job_id"]
print(supply_chain)

In [ ]:
time.sleep(10)

# Supplier → Product → Warehouse chains
results = run_cypher("""
    MATCH (s:Supplier)-[sup:SUPPLIES]->(p:Product)-[st:STORED_IN]->(w:Warehouse)
    RETURN s.name AS supplier, s.country AS country, s.reliability_score AS reliability,
           p.sku AS sku, p.category AS category, sup.contract_price AS price,
           w.location AS warehouse, st.quantity AS stock
    ORDER BY st.quantity DESC
    LIMIT 25
""")
if results:
    display(spark.createDataFrame(results))
else:
    print("No supply chains found — relationships may not have connected")

### Write Neo4j Custom Data to Delta

In [ ]:
# Export org chart to Delta
employees = run_cypher("""
    MATCH (e:Employee)
    OPTIONAL MATCH (e)-[:WORKS_IN]->(o:Office)
    RETURN e.emp_id AS emp_id, e.name AS name, e.department AS department,
           e.title AS title, e.salary AS salary, e.hire_date AS hire_date,
           e.remote AS remote, o.city AS office_city
""")
if employees:
    df = spark.createDataFrame(employees)
    df.write.format("delta").mode("overwrite").saveAsTable(f"{DELTA_BASE}.custom_employees")
    print(f"Wrote {df.count()} employees to {DELTA_BASE}.custom_employees")

---
# Part 2: Custom PostgreSQL Tables

---
## Example 1: Customer Survey Responses

In [ ]:
survey = requests.post(f"{API_URL}/data-sources/custom/start", headers=HEADERS, json={
    "name": "customer_survey",
    "table_name": "survey_responses",
    "num_records": 10000,
    "drop_existing": True,
    "columns": [
        {"name": "response_id", "sql_type": "VARCHAR(40)", "primary_key": True,
         "generator_rule": {"generator": "uuid"}},
        {"name": "respondent", "sql_type": "VARCHAR(100)",
         "generator_rule": {"generator": "name"}},
        {"name": "email", "sql_type": "VARCHAR(120)",
         "generator_rule": {"generator": "email", "domain": "survey-users.com"}},
        {"name": "overall_score", "sql_type": "INT",
         "generator_rule": {"generator": "range_int", "min": 1, "max": 10}},
        {"name": "nps_score", "sql_type": "INT",
         "generator_rule": {"generator": "range_int", "min": 0, "max": 10}},
        {"name": "category", "sql_type": "VARCHAR(20)",
         "generator_rule": {"generator": "choice", "values": ["promoter", "passive", "detractor"], "weights": [5, 3, 2]}},
        {"name": "product", "sql_type": "VARCHAR(30)",
         "generator_rule": {"generator": "choice", "values": ["Platform", "Mobile App", "API", "Dashboard", "Analytics"]}},
        {"name": "submitted_at", "sql_type": "DATE",
         "generator_rule": {"generator": "date", "start": "2024-01-01", "end": "2025-12-31"}},
        {"name": "country", "sql_type": "VARCHAR(5)",
         "generator_rule": {"generator": "choice", "values": ["US", "UK", "DE", "FR", "CA", "AU", "JP", "BR"]}},
    ],
}).json()

survey_job_id = survey["job_id"]
print(survey)

In [ ]:
time.sleep(5)

# Check schema
schema = requests.get(f"{API_URL}/data-sources/custom/{survey_job_id}/schema", headers=HEADERS).json()
print(f"Table: {schema['table']} — {schema.get('row_count', 0)} rows")
for col in schema.get("columns", []):
    print(f"  {col['column_name']:20s} {col['data_type']:20s} nullable={col['is_nullable']}")

In [ ]:
# Sample rows
sample = requests.get(f"{API_URL}/data-sources/custom/{survey_job_id}/sample?limit=5", headers=HEADERS).json()
for row in sample.get("rows", []):
    print(row)

In [ ]:
# Read via JDBC and analyze
df_survey = read_pg_table("custom_survey_responses")
display(
    df_survey
    .groupBy("product", "category")
    .agg(
        count("*").alias("responses"),
        avg("overall_score").alias("avg_score"),
        avg("nps_score").alias("avg_nps"),
    )
    .orderBy(col("responses").desc())
)

---
## Example 2: Fleet Vehicle Tracking

In [ ]:
fleet = requests.post(f"{API_URL}/data-sources/custom/start", headers=HEADERS, json={
    "name": "fleet_tracking",
    "table_name": "vehicle_events",
    "num_records": 20000,
    "drop_existing": True,
    "columns": [
        {"name": "event_id", "sql_type": "VARCHAR(40)", "primary_key": True,
         "generator_rule": {"generator": "uuid"}},
        {"name": "vehicle_id", "sql_type": "VARCHAR(12)",
         "generator_rule": {"generator": "sequence", "prefix": "VH-", "width": 5}},
        {"name": "driver", "sql_type": "VARCHAR(100)",
         "generator_rule": {"generator": "name"}},
        {"name": "vehicle_type", "sql_type": "VARCHAR(20)",
         "generator_rule": {"generator": "choice", "values": ["sedan", "suv", "van", "truck", "bus"], "weights": [4, 3, 2, 3, 1]}},
        {"name": "event_type", "sql_type": "VARCHAR(20)",
         "generator_rule": {"generator": "choice",
             "values": ["ignition_on", "ignition_off", "speeding", "hard_brake", "idle", "geofence_enter", "geofence_exit", "maintenance_due"],
             "weights": [4, 4, 2, 2, 3, 1, 1, 1]}},
        {"name": "speed_mph", "sql_type": "INT",
         "generator_rule": {"generator": "range_int", "min": 0, "max": 95}},
        {"name": "fuel_pct", "sql_type": "DECIMAL(5,2)",
         "generator_rule": {"generator": "range_float", "min": 5.0, "max": 100.0}},
        {"name": "odometer_miles", "sql_type": "INT",
         "generator_rule": {"generator": "range_int", "min": 1000, "max": 200000}},
        {"name": "latitude", "sql_type": "DECIMAL(8,5)",
         "generator_rule": {"generator": "range_float", "min": 25.0, "max": 48.0, "precision": 5}},
        {"name": "longitude", "sql_type": "DECIMAL(9,5)",
         "generator_rule": {"generator": "range_float", "min": -125.0, "max": -70.0, "precision": 5}},
        {"name": "event_time", "sql_type": "TIMESTAMPTZ",
         "generator_rule": {"generator": "timestamp", "start": "2024-06-01T00:00:00", "end": "2025-12-31T23:59:59"}},
    ],
}).json()

fleet_job_id = fleet["job_id"]
print(fleet)

In [ ]:
time.sleep(10)

df_fleet = read_pg_table("custom_vehicle_events")
print(f"Vehicle events: {df_fleet.count():,} rows")

display(
    df_fleet
    .groupBy("vehicle_type", "event_type")
    .agg(
        count("*").alias("events"),
        avg("speed_mph").alias("avg_speed"),
        avg("fuel_pct").alias("avg_fuel"),
    )
    .orderBy(col("events").desc())
)

### Write Custom Postgres Data to Delta

In [ ]:
# Write survey data to Delta
df_survey = read_pg_table("custom_survey_responses")
df_survey.write.format("delta").mode("overwrite").saveAsTable(f"{DELTA_BASE}.custom_survey_responses")
print(f"Wrote {df_survey.count():,} rows to {DELTA_BASE}.custom_survey_responses")

# Write fleet data to Delta
df_fleet = read_pg_table("custom_vehicle_events")
df_fleet.write.format("delta").mode("overwrite").saveAsTable(f"{DELTA_BASE}.custom_vehicle_events")
print(f"Wrote {df_fleet.count():,} rows to {DELTA_BASE}.custom_vehicle_events")

---
## Management — List Jobs, Inspect, Drop

In [ ]:
# List all custom Neo4j jobs
print("Neo4j custom jobs:")
for job in requests.get(f"{API_URL}/data-sources/neo4j/custom", headers=HEADERS).json():
    print(f"  {job['name']} ({job['job_id']}): {job['status']} — nodes: {job.get('nodes_created', {})}")

# List all custom Postgres jobs
print("\nPostgres custom jobs:")
for job in requests.get(f"{API_URL}/data-sources/custom", headers=HEADERS).json():
    print(f"  {job['name']} ({job['job_id']}): {job['status']} — {job.get('rows_created', 0)} rows in {job['table_name']}")

---
## Cleanup

In [ ]:
driver.close()
print("Neo4j driver closed")

# Uncomment to clear/drop everything created in this notebook:

# Clear Neo4j custom data
# for job in requests.get(f"{API_URL}/data-sources/neo4j/custom", headers=HEADERS).json():
#     resp = requests.delete(f"{API_URL}/data-sources/neo4j/custom/{job['job_id']}/clear", headers=HEADERS)
#     print(f"Cleared Neo4j: {resp.json()}")

# Drop Postgres custom tables
# for job in requests.get(f"{API_URL}/data-sources/custom", headers=HEADERS).json():
#     resp = requests.delete(f"{API_URL}/data-sources/custom/{job['job_id']}/drop", headers=HEADERS)
#     print(f"Dropped: {resp.json()}")

# Clean up job lists
# requests.delete(f"{API_URL}/data-sources/neo4j/custom/cleanup", headers=HEADERS)
# requests.delete(f"{API_URL}/data-sources/custom/cleanup", headers=HEADERS)